Подготовка модулей и загрузка моделей

In [3]:
from sentence_transformers import SentenceTransformer, util
import json
import numpy as np
import pandas as pd

Инициализация двух моделей

In [4]:
model1 = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
model2 = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Выгрузка данных из JSON и эмбеддинг файла code_corpus.json

In [5]:
with open('code_corpus.json', 'r', encoding='utf-8') as f:
    data_embedding_code_corpus = json.load(f)

embeddings1_code_corpus = []
embeddings2_code_corpus = []
text = None

for i in data_embedding_code_corpus:
    text = json.dumps(i, ensure_ascii=False, sort_keys=True)

    embeddings1_code_corpus.append(model1.encode(text))
    embeddings2_code_corpus.append(model2.encode(text))

Выгрузка данных из JSON и эмбеддинг файла eval_questions.json. Срез 

In [6]:
with open('eval_questions.json', 'r', encoding='utf-8') as j:
    data_eval_questions = json.load(j)

embeddings1_eval_questions = []
embeddings2_eval_questions = []
count = 0
text = None

for u in data_eval_questions:
    if count >= 15: break
    count += 1
    text = json.dumps(u, ensure_ascii=False, sort_keys=True)

    embeddings1_eval_questions.append(model1.encode(text))
    embeddings2_eval_questions.append(model2.encode(text))

In [7]:
np_code_corpus1 = np.array(embeddings1_code_corpus)
np_code_corpus2 = np.array(embeddings2_code_corpus)

np_eval_questions1 = np.array(embeddings1_eval_questions)
np_eval_questions2 = np.array(embeddings2_eval_questions)

total_cos_similarity1 = []
total_cos_similarity2 = []

for h in np_eval_questions1:
    cos_similarity1 = []
    for r in np_code_corpus1:
        cos_similarity1.append(util.cos_sim(h, r).item())
    total_cos_similarity1.append(cos_similarity1)

for s in np_eval_questions2:
    cos_similarity2 = []
    for d in np_code_corpus2:
        cos_similarity2.append(util.cos_sim(s, d).item())
    total_cos_similarity2.append(cos_similarity2)

# df = pd.DataFrame(total_cos_similarity1)
# df.to_csv("debug/total_cos_similarity1.csv")
#
# df = pd.DataFrame(total_cos_similarity2)
# df.to_csv("debug/total_cos_similarity2.csv")

In [30]:
np_total_cos_similarity1 = np.array(total_cos_similarity1)
np_total_cos_similarity2 = np.array(total_cos_similarity2)

top3_output_index_1 = np_total_cos_similarity1.argsort(axis=1)[:,::-1][:,:3]
top3_output_index_2 = np_total_cos_similarity2.argsort(axis=1)[:,::-1][:,:3]

all_correct1 = []
all_correct2 = []

precision_3_1 = 0.0
precision_3_2 = 0.0

for i in range(len(embeddings1_eval_questions)):
    correct_list = None
    for j in top3_output_index_1[i]:
        if data_embedding_code_corpus[j]['id'] == data_eval_questions[i]['correct_chunk_id']:
            correct_list = True
        if not correct_list:
            correct_list = False
    all_correct1.append(correct_list)

for i in range(len(embeddings2_eval_questions)):
    correct_list = None
    for j in top3_output_index_2[i]:
        if data_embedding_code_corpus[j]['id'] == data_eval_questions[i]['correct_chunk_id']:
            correct_list = True
        if not correct_list:
            correct_list = False
    all_correct2.append(correct_list)

precision_3_1 = all_correct1.count(True) / len(all_correct1)
precision_3_2 = all_correct2.count(True) / len(all_correct2)            # Лучшей моделью оказалась модель №2

